# ARCHS4 — Tissue enrichment in CLAMP LVs (pancreas, liver, brain)

**Environment:** `clamp-analyses`

Uses the **B matrix** (sample loadings) from the archs4 CLAMPfull C2CP model to identify\nLVs whose top-loaded samples are significantly enriched for a given tissue type.

Tissue sample accessions are retrieved from the ARCHS4 h5 file using the `archs4r` package.\n\nFor each LV:
1. Samples are ranked by B loading (descending)
2. The **top 1% of samples** (highest loading) are taken as the foreground
3. Enrichment of tissue samples in that foreground is tested with a one-sided Wilcoxon rank-sum AUC (tissue vs. background)

LVs are ranked by AUC and filtered by FDR.\n\nTissues: **pancreas**, **liver**, **brain**\n\nInputs:
- `output/archs4/archs4_CLAMP_C2CP.rds`
- `data/archs4/human_gene_v2.5.h5`

## Libraries

In [14]:
library(archs4r)
library(here)
library(dplyr)
library(ggplot2)
library(tidyr)

source(here("config.R"))

## Load data

In [15]:
h5file  <- here("data", "archs4", "human_gene_v2.5.h5")
rds_in  <- here("output", "archs4", "archs4_CLAMP_C2CP.rds")

stopifnot(file.exists(h5file))
stopifnot(file.exists(rds_in))

archs4_CLAMPfull <- readRDS(rds_in)

message("Model loaded")

Model loaded



In [16]:
# B matrix: rows = LVs, cols = sample accessions
B_mat <- as.matrix(archs4_CLAMPfull$B)
rownames(B_mat) <- paste0("LV", seq_len(nrow(B_mat)))
all_samples <- colnames(B_mat)

n_lvs  <- nrow(B_mat)
n_samp <- ncol(B_mat)

message("B matrix : ", n_lvs, " LVs x ", n_samp, " samples")
message("Sample ID example: ", head(all_samples, 3))

# Summary and Z for later annotation
summary_df <- archs4_CLAMPfull$summary
if (!is.data.frame(summary_df)) summary_df <- data.frame(as.matrix(summary_df))
summary_df <- summary_df %>%
    dplyr::mutate(FDR = as.numeric(FDR), AUC = as.numeric(AUC))

# Free expression data to save memory
archs4_CLAMPfull$Z <- NULL
archs4_CLAMPfull$B <- NULL
gc()

B matrix : 2366 LVs x 605614 samples

Sample ID example: GSM1000981GSM1000982GSM1000983



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2027517,108.3,5907675,315.6,9563266,510.8
Vcells,1662425059,12683.3,5491718723,41898.5,4577489649,34923.5


In [ ]:
head(B_mat)

,GSM1000981,GSM1000982,GSM1000983,GSM1000984,GSM1000985,GSM1000986,GSM1002540,GSM1002541,GSM1002542,GSM1002543,⋯,GSM5586690,GSM5586691,GSM5586692,GSM5586693,GSM5586694,GSM5586695,GSM5586696,GSM5586697,GSM5586698,GSM5586699
LV1,0.17383585,0.17684266,0.18158742,0.18171956,0.17988606,0.18036345,-0.201187240,-0.204439185,-0.1929257608,-0.210662928,⋯,-0.14395115,-0.09269412,-0.08287168,-0.10168084,-0.15471213,-0.12977420,-0.09719972,-0.12320250,-0.262556047,-0.11660257
LV2,-0.04774563,-0.04386446,-0.03994312,-0.02906579,-0.03470205,-0.02623380,-0.166700474,-0.179441687,-0.1839082002,-0.184115057,⋯,-0.05443984,-0.03790012,-0.06165449,-0.06934788,-0.03380297,-0.09542965,-0.04704147,-0.02565607,-0.181879728,-0.02334972
LV3,-0.14761690,-0.14607387,-0.14829155,-0.14253644,-0.14213225,-0.14229245,-0.036584940,-0.022929024,-0.0379376927,-0.033625721,⋯,0.12607497,0.14546716,0.17148073,0.14337144,0.09908379,0.10264401,0.16129592,0.13742582,-0.031573424,0.13406128
LV4,0.12131664,0.11571741,0.12173255,0.11667320,0.10627004,0.11334892,-0.198182921,-0.202222815,-0.1887861777,-0.204930877,⋯,-0.11503054,-0.07781242,-0.05392682,-0.06118665,-0.15085958,-0.10826778,-0.07901089,-0.11014336,-0.233313734,-0.11263464
LV5,-0.08321888,-0.08179165,-0.08081104,-0.07550733,-0.07393693,-0.07685731,0.001514206,0.009858751,0.0078031329,-0.001727206,⋯,-0.03680436,-0.03629894,-0.04042323,-0.04352608,-0.02429167,-0.03366445,-0.04344188,-0.02850851,-0.006776316,-0.04008374
LV6,-0.08262342,-0.08333921,-0.08261051,-0.07188577,-0.06744781,-0.07322516,-0.001968661,0.006808704,-0.0006042821,-0.004544787,⋯,0.14496459,0.16512689,0.16270740,0.17823915,0.11769516,0.13561777,0.16780662,0.13718452,-0.020278737,0.15014696


: 

## Retrieve tissue samples from ARCHS4 h5

In [ ]:
tissues <- c("pancreas", "liver", "brain")

tissue_samples <- list()
tissue_meta    <- list()

for (tissue in tissues) {
    message("Querying ARCHS4 for: ", tissue)

    # Step 1: get expression matrix for samples matching the tissue keyword
    # (colnames are internal h5 sample IDs used by archs4r)
    tissue_exp <- a4.data.meta(
        h5file,
        tissue,
        c("title", "source_name_ch1", "characteristics_ch1"),
        remove_sc = TRUE
    )
    sample_ids <- colnames(tissue_exp)
    rm(tissue_exp)
    gc()

    # Step 2: retrieve metadata for those samples, including geo_accession
    # a4.meta.samples returns a data frame: rows = samples, cols = metadata fields
    meta <- a4.meta.samples(
        h5file,
        sample_ids,
        c("geo_accession", "title", "source_name_ch1", "series_id", "characteristics_ch1")
    )
    tissue_meta[[tissue]] <- meta

    # Step 3: use geo_accession (GSM IDs) to intersect with B matrix colnames
    geo_acc  <- meta[, "geo_accession"]
    in_model <- intersect(geo_acc, all_samples)
    tissue_samples[[tissue]] <- in_model

    message(sprintf("  %-10s : %d in h5 | %d geo_acc | %d in B matrix",
                    tissue, length(sample_ids), length(geo_acc), length(in_model)))
    message("  sample_ids[:3] : ", paste(head(sample_ids, 3), collapse = ", "))
    message("  geo_acc[:3]    : ", paste(head(geo_acc, 3), collapse = ", "))
    message("  B mat IDs[:3]  : ", paste(head(all_samples, 3), collapse = ", "))
}

Querying ARCHS4 for: pancreas



Searching for any occurrence of pancreas as regular expression
Extracting field 'geo_accession' for 1969 samples
Extracting field 'title' for 1969 samples
Extracting field 'source_name_ch1' for 1969 samples
Extracting field 'series_id' for 1969 samples
Extracting field 'characteristics_ch1' for 1969 samples


  pancreas   : 1969 in h5 | 1969 geo_acc | 1324 in B matrix

  sample_ids[:3] : GSM1129243, GSM1129244, GSM1129245

  geo_acc[:3]    : GSM1129243, GSM1129244, GSM1129245

  B mat IDs[:3]  : GSM1000981, GSM1000982, GSM1000983

Querying ARCHS4 for: liver



In [ ]:
head(ids)

[1] "LSEC"                                                                    
[2] "Biochain:R-1234149-P"                                                    
[3] "Strategene Universal Human Reference RNA (UHRR) from 10 human cell lines"
[4] "metastasized cancer"                                                     
[5] "liver, postnatal"                                                        
[6] "freshly isolated human hepatocytes"

## Tissue enrichment in B matrix

For each LV, samples are ranked by B loading (descending).
Enrichment of tissue samples is measured via the Wilcoxon rank-sum AUC
(one-sided: tissue > background). P-values use the normal approximation.
FDR is Benjamini-Hochberg across all LVs within each tissue.

In [ ]:
# Returns a data.frame with AUC, p-value, FDR, and top-1% count for every LV
compute_tissue_enrichment <- function(B_mat, tissue_ids, top_pct = 0.01) {

    tissue_idx <- which(colnames(B_mat) %in% tissue_ids)
    n1 <- length(tissue_idx)
    n2 <- ncol(B_mat) - n1
    n_top <- max(1L, round(ncol(B_mat) * top_pct))

    if (n1 == 0) stop("No tissue samples found in B matrix.")

    mu_U    <- n1 * n2 / 2
    sigma_U <- sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

    message("Computing enrichment for ", nrow(B_mat), " LVs ",
            "(", n1, " tissue vs ", n2, " background samples) ...")

    result <- t(apply(B_mat, 1, function(b) {
        r         <- rank(b, ties.method = "average")
        rank_sum  <- sum(r[tissue_idx])
        U         <- rank_sum - n1 * (n1 + 1) / 2
        auc       <- U / (n1 * n2)
        z         <- (U - mu_U) / sigma_U
        pval      <- pnorm(z, lower.tail = FALSE)

        top_cols  <- order(b, decreasing = TRUE)[seq_len(n_top)]
        n_in_top  <- sum(top_cols %in% tissue_idx)

        c(AUC = auc, pvalue = pval, n_tissue_in_top1pct = n_in_top)
    }))

    df <- as.data.frame(result)
    df$LV     <- rownames(B_mat)
    df$FDR    <- p.adjust(df$pvalue, method = "BH")
    df$n_top  <- n_top
    df$n_tissue <- n1

    dplyr::select(df, LV, AUC, pvalue, FDR, n_tissue_in_top1pct, n_top, n_tissue)
}

In [ ]:
enrichment_results <- list()

for (tissue in tissues) {
    message("\n", strrep("-", 50))
    message("Tissue: ", tissue)
    enrichment_results[[tissue]] <- compute_tissue_enrichment(
        B_mat,
        tissue_samples[[tissue]],
        top_pct = 0.01
    )
    message("Done.")
}


--------------------------------------------------

Tissue: pancreas



ERROR: Error in compute_tissue_enrichment(B_mat, tissue_samples[[tissue]], top_pct = 0.01): No tissue samples found in B matrix.


## LVs enriched for tissue samples in top 1% loading
For each LV the top 1% of samples by B loading are tested for tissue enrichment.
All LVs are shown ranked by AUC (descending); use FDR < 0.05 to highlight significant ones.

In [ ]:
FDR_CUTOFF <- 0.05

top_lvs <- list()

for (tissue in tissues) {
    top_lvs[[tissue]] <- enrichment_results[[tissue]] %>%
        dplyr::arrange(dplyr::desc(AUC)) %>%
        dplyr::mutate(tissue = tissue)

    n_sig <- sum(top_lvs[[tissue]]$FDR < FDR_CUTOFF)
    message(tissue, ":", n_sig, " LVs with FDR < ", FDR_CUTOFF,
            "  (top-1%-sample AUC n_top = ",
            unique(top_lvs[[tissue]]$n_top), " samples)")

    print(
        top_lvs[[tissue]] %>%
            dplyr::filter(FDR < FDR_CUTOFF) %>%
            dplyr::select(LV, AUC, FDR, n_tissue_in_top1pct, n_top, n_tissue)
    )
}

## Pathway annotations for top LVs

In [ ]:
# Significant C2CP pathways (FDR < 0.05) per LV
sig_pathways <- summary_df %>%
    dplyr::filter(FDR < 0.05) %>%
    dplyr::group_by(LV) %>%
    dplyr::summarise(
        top_pathway = pathway[which.max(AUC)],
        n_sig_paths = dplyr::n(),
        max_AUC_pw  = max(AUC),
        .groups = "drop"
    )

for (tissue in tissues) {
    message("\n── ", toupper(tissue), " ──")

    annotated <- top_lvs[[tissue]] %>%
        dplyr::left_join(sig_pathways, by = "LV") %>%
        dplyr::select(LV, AUC, FDR, n_tissue_in_top1pct,
                      n_sig_paths, top_pathway, max_AUC_pw)

    print(annotated)
}

## Visualisation — top 1% LVs per tissue

In [ ]:
tissue_colors <- c(pancreas = "#E69F00", liver = "#56B4E9", brain = "#009E73")

plot_top_lvs <- function(df, tissue_name, fdr_cutoff = 0.05) {
    df <- df %>%
        dplyr::filter(FDR < fdr_cutoff) %>%
        dplyr::arrange(dplyr::desc(AUC)) %>%
        dplyr::mutate(LV = factor(LV, levels = rev(LV)))

    if (nrow(df) == 0) {
        message(tissue_name, ": no FDR-significant LVs to plot.")
        return(invisible(NULL))
    }

    col <- tissue_colors[[tissue_name]]

    ggplot(df, aes(x = AUC, y = LV)) +
        geom_segment(aes(x = 0.5, xend = AUC, y = LV, yend = LV),
                     colour = "#cccccc", linewidth = 0.4, linetype = "dotted") +
        geom_point(aes(size = n_tissue_in_top1pct),
                   fill = col, color = "#333333", shape = 21,
                   stroke = 0.4, alpha = 0.9) +
        geom_vline(xintercept = 0.5, linetype = "dashed",
                   colour = "#888888", linewidth = 0.3) +
        scale_size_continuous(name = "# tissue samples\\nin top 1% of B",
                              range = c(3, 10)) +
        labs(
            title   = paste0("FDR-significant LVs — ", tools::toTitleCase(tissue_name)),
            subtitle = paste0("Enrichment based on top 1% of samples by B loading (",
                              unique(df$n_top), " samples)"),
            x = "AUC (tissue vs. background)",
            y = NULL
        ) +
        theme_bw(base_size = 12) +
        theme(
            panel.grid.minor   = element_blank(),
            panel.grid.major.y = element_blank(),
            axis.text.y        = element_text(size = 9)
        )
}

options(repr.plot.width = 8, repr.plot.height = 7)
for (tissue in tissues) {
    p <- plot_top_lvs(top_lvs[[tissue]], tissue)
    if (!is.null(p)) print(p)
}

## AUC heatmap across tissues

In [ ]:
# Union of FDR-significant LVs across all tissues
union_lvs <- unique(unlist(lapply(top_lvs, function(df) {
    df$LV[df$FDR < FDR_CUTOFF]
})))

if (length(union_lvs) == 0) {
    message("No FDR-significant LVs found — relaxing to top 30 by AUC per tissue.")
    union_lvs <- unique(unlist(lapply(top_lvs, function(df) {
        head(df$LV, 30)
    })))
}

heatmap_df <- dplyr::bind_rows(lapply(tissues, function(tissue) {
    enrichment_results[[tissue]] %>%
        dplyr::filter(LV %in% union_lvs) %>%
        dplyr::mutate(tissue = tissue)
})) %>%
    dplyr::mutate(
        LV     = factor(LV, levels = union_lvs),
        tissue = factor(tissue, levels = tissues)
    )

options(repr.plot.width = 11, repr.plot.height = max(5, length(union_lvs) * 0.35))

ggplot(heatmap_df, aes(x = tissue, y = LV, fill = AUC)) +
    geom_tile(colour = "white", linewidth = 0.4) +
    geom_text(aes(label = ifelse(FDR < FDR_CUTOFF, "*", "")),
              size = 4, colour = "white", fontface = "bold") +
    scale_fill_gradient2(
        low = "#2166ac", mid = "white", high = "#d73027",
        midpoint = 0.5, limits = c(0.4, 1),
        name = "AUC"
    ) +
    labs(
        title   = "Tissue enrichment AUC — FDR-significant LVs (union across tissues)",
        caption = paste0("* FDR < ", FDR_CUTOFF, "; enrichment based on top 1% of samples per LV"),
        x = NULL, y = NULL
    ) +
    theme_bw(base_size = 12) +
    theme(
        axis.text.x   = element_text(size = 11, face = "bold"),
        axis.text.y   = element_text(size = 8),
        panel.grid    = element_blank()
    )

## Save results

In [ ]:
out_dir <- here("output", "archs4", "tissue_enrichment")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

for (tissue in tissues) {
    write.csv(
        enrichment_results[[tissue]],
        file.path(out_dir, paste0(tissue, "_enrichment.csv")),
        row.names = FALSE
    )
    write.csv(
        top_lvs[[tissue]],
        file.path(out_dir, paste0(tissue, "_top1pct_lvs.csv")),
        row.names = FALSE
    )
}

message("Results saved to ", out_dir)